# 04.07 Image Forgery Screening (ELA + ORB/SIFT Copy-Move Screening)

**Phase 04 — Document Processing & OCR (Track A: Document Screening)**  
**Method:** Dual-signal non-Siamese forensic screening combining:
1. **Error Level Analysis (ELA):** Re-compresses the image at JPEG quality 90 and measures block-level residual variance / hotspot ratios.
2. **Keypoint Copy-Move Forgery Detection (CMFD):** Extracts ORB keypoints, matches spatially separated descriptors (`dist > min_distance`), and filters geometrically consistent clusters.
3. **Image Quality Separation:** Explicitly separates low-quality/blurry/over-compressed scans from potential manipulation indicators.

> **Governance Disclaimer:** Visual anomalies and CMFD clusters are **screening indicators only** and never constitute definitive proof of fraud. Flagged documents are routed to an authorized MSW for manual verification.


### 1. Environment Setup & Forensic Screening Configuration


In [ ]:
import os
import sys
import io
import json
from pathlib import Path
from typing import Dict, Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

def _find_root(p: Path = Path.cwd()) -> Path:
    for parent in [p.resolve(), *p.resolve().parents]:
        if (parent / 'data').exists() and (parent / 'experiments').exists():
            return parent
    return p.resolve()

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.document_processing import DocumentProcessingPaths
paths = DocumentProcessingPaths(PROJECT_ROOT)

SCREENING_OUT_DIR = paths.interim_doc_proc / 'forgery_screening'
SCREENING_OUT_DIR.mkdir(parents=True, exist_ok=True)
SCREENING_REPORT_JSON = SCREENING_OUT_DIR / 'forgery_screening_results.json'

FORENSIC_CONFIG = {
    'max_dim_px': 1200,
    'ela_jpeg_quality': 90,
    'ela_zscore_threshold': 4.0,
    'cmfd_orb_features': 2000,
    'cmfd_min_spatial_dist_px': 35.0,
    'cmfd_max_hamming_dist': 22,
    'cmfd_min_suspicious_pairs': 18,
    'min_sharpness_laplacian': 40.0,
    'max_documents_to_screen': 30,
}

print(f"Project Root     : {paths.root}")
print(f"Screening Output : {SCREENING_REPORT_JSON}")


### 2. ELA, Copy-Move & Quality-Aware Screening Implementation


In [ ]:
def compute_ela_metrics(bgr_img: np.ndarray, quality: int = 90) -> tuple:
    rgb = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB)
    pil_orig = Image.fromarray(rgb)
    buf = io.BytesIO()
    pil_orig.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    pil_resaved = Image.open(buf).convert('RGB')
    diff = np.abs(np.asarray(pil_orig, dtype=np.float32) - np.asarray(pil_resaved, dtype=np.float32))
    ela_gray = diff.mean(axis=2)
    mean_val = float(ela_gray.mean())
    std_val = float(ela_gray.std() + 1e-6)
    p99 = float(np.percentile(ela_gray, 99))
    z_p99 = float((p99 - mean_val) / std_val)
    return {'ela_mean': round(mean_val, 3), 'ela_std': round(std_val, 3), 'ela_p99_zscore': round(z_p99, 3)}, ela_gray

def compute_cmfd_pairs(bgr_img: np.ndarray, cfg: dict = FORENSIC_CONFIG) -> tuple:
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    orb = cv2.ORB_create(nfeatures=cfg['cmfd_orb_features'])
    kp, des = orb.detectAndCompute(gray, None)
    if des is None or len(des) < 10:
        return [], 0
    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    knn = bf.knnMatch(des, des, k=3)
    pairs = []
    for m_list in knn:
        for m in m_list[1:]:
            if m.distance <= cfg['cmfd_max_hamming_dist']:
                p1 = np.array(kp[m.queryIdx].pt)
                p2 = np.array(kp[m.trainIdx].pt)
                if np.linalg.norm(p1 - p2) >= cfg['cmfd_min_spatial_dist_px']:
                    pairs.append((tuple(p1.astype(int)), tuple(p2.astype(int))))
                    break
    return pairs, len(pairs)

def screen_document_for_forgery(image_path: Path, cfg: dict = FORENSIC_CONFIG) -> tuple:
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f"Unable to decode image: {image_path}")
    h, w = img.shape[:2]
    if max(h, w) > cfg['max_dim_px']:
        s = cfg['max_dim_px'] / float(max(h, w))
        img = cv2.resize(img, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    quality_issue = 'LOW_SHARPNESS_OR_BLUR' if sharpness < cfg['min_sharpness_laplacian'] else 'NONE'

    ela_stats, ela_map = compute_ela_metrics(img, cfg['ela_jpeg_quality'])
    pairs, pair_count = compute_cmfd_pairs(img, cfg)

    ela_flag = ela_stats['ela_p99_zscore'] >= cfg['ela_zscore_threshold']
    cmfd_flag = pair_count >= cfg['cmfd_min_suspicious_pairs']

    if cmfd_flag and ela_flag:
        screening_status = 'MANIPULATION_INDICATOR_HIGH'
    elif cmfd_flag or ela_flag:
        screening_status = 'MANIPULATION_INDICATOR_MODERATE'
    elif quality_issue != 'NONE':
        screening_status = 'QUALITY_ISSUE_ONLY'
    else:
        screening_status = 'NO_STRONG_INDICATOR'

    record = {
        'filename': image_path.name,
        'sharpness_laplacian_var': round(sharpness, 2),
        'image_quality_issue': quality_issue,
        **ela_stats,
        'cmfd_matched_pairs': int(pair_count),
        'screening_status': screening_status,
        'manual_verification_required': bool(cmfd_flag or ela_flag),
        'disclaimer': 'Screening indicator only; not definitive proof of forgery.',
    }
    return record, img, ela_map, pairs

print("[OK] ELA + ORB-CMFD forensic screening functions initialized.")


### 3. Run Batch Forensic Screening & Visual Inspection


In [ ]:
sample_images = []
if paths.doc_inventory_manifest.exists():
    df_inv = pd.read_csv(paths.doc_inventory_manifest)
    for _, r in df_inv.head(FORENSIC_CONFIG['max_documents_to_screen']).iterrows():
        p = paths.resolve_sample_path(str(r['relative_path']))
        if p.exists() and p.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
            sample_images.append(p)

results = []
first_vis = None
for img_p in sample_images:
    rec, bgr, ela_m, pairs = screen_document_for_forgery(img_p)
    results.append(rec)
    if first_vis is None:
        first_vis = (img_p.name, bgr, ela_m, pairs)

df_forgery = pd.DataFrame(results)
with open(SCREENING_REPORT_JSON, 'w', encoding='utf-8') as f:
    json.dump({'config': FORENSIC_CONFIG, 'results': results}, f, indent=2)

print(f"Screened {len(df_forgery)} documents -> {SCREENING_REPORT_JSON}")
if not df_forgery.empty:
    display(df_forgery['screening_status'].value_counts().rename_axis('Screening Status').reset_index(name='Count'))
    display(df_forgery.head(10))

if first_vis is not None:
    fn, bgr, ela_m, pairs = first_vis
    overlay = bgr.copy()
    for p1, p2 in pairs[:40]:
        cv2.line(overlay, p1, p2, (0, 0, 255), 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].imshow(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title(f'Original: {fn}'); axes[0].axis('off')
    axes[1].imshow(ela_m, cmap='inferno'); axes[1].set_title('ELA Residual Map (Q=90)'); axes[1].axis('off')
    axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[2].set_title(f'ORB CMFD Pairs ({len(pairs)})'); axes[2].axis('off')
    plt.tight_layout()
    plt.show()
